# 5.0 Agents for Item Generation

What an agent is, what a workflow is, when you need each, and how to build both for item generation.

| | |
|---|---|
| **Time** | ~45 minutes |
| **Keys needed** | `GROQ_API_KEY` |

**In this notebook**

1. What is an agent?
2. What is a workflow?
3. When do you need an agent?
4. Build a workflow in three steps: write → review → check → human queue
5. Build an agent: a Repair Agent that fixes items the workflow held back

## 1. What is an agent?

An agent is **a model that calls tools in a loop**. You give it a goal and a set of tools. It decides
which tool to call, reads the result, and decides again. It stops when it judges the goal is met, or
when a limit you set stops it.

```
goal → [model decides] → tool → result → [model decides] → tool → result → ... → done
```

The defining feature: **the model chooses the next step**, so you do not know in advance how many
steps it will take or which ones.

A **tool** is an ordinary Python function the model may *ask* you to run. The model never runs anything
itself; it emits a request, and your code runs the function and hands back the result. What the model
sees is the function's name, its docstring, and its argument names and types. The docstring is a
prompt; the signature is a contract.

## 2. What is a workflow?

A workflow is **a set of model calls arranged in an order your code decides**. Each call can be a
specialist (a writer, a reviewer), and people often call these specialists "agents."
What makes it a workflow is that **the path is fixed**.

```
objective → [Writer] → [Reviewer] → [Rules] → human queue
```

Most multi-agent item-generation systems in operational use are workflows: specialist roles, a fixed
order, a schema at every handoff. That is a strength, not a compromise. A workflow is reproducible,
each stage can be tested on its own, and when something fails you know which stage failed.

## 3. When do you need an agent?

> **If you can draw the steps before you run them, build a workflow.**
> **If the model has to decide the next step from what it just found, use an agent.**

| Situation in item development | Use |
|---|---|
| Generate items for a blueprint cell or item model | Workflow |
| Run the same quality checks on every item | Workflow |
| You need a reproducible, auditable process | Workflow |
| Fix a flawed item, where the number and kind of fixes vary item to item | **Agent** |
| Research open-ended source material before writing | **Agent** |

**Note**:

- **Agents cost more.** They are slower, less reproducible, and their errors can compound across
  steps. Start with a workflow and add an agent only where the workflow keeps falling short.

## Setup

Run these two cells once. You only need a Groq key saved in Colab Secrets as `GROQ_API_KEY`.

In [ ]:
%pip install -q -U langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.6 MB/s eta 0:00:00


In [ ]:
import os, re
from typing import Literal
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    print("Could not read Colab Secrets. Check the key name and Notebook access.")

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)


def safe(fn, *args):
    """Run a step; if the model call fails, say why in one line instead of a long error."""
    try:
        return fn(*args)
    except Exception as e:
        hint = "wait 60 seconds (rate limit)" if "429" in str(e) else "check your key, then re-run"
        print(f"Call failed: {type(e).__name__}. Try: {hint}.")


print("Ready.")

Ready.


### The use case

Overarching Goal: **write a grade 8 multiple-choice question for a learning objective, and make sure it is good enough for a person to review.**


In [ ]:
OBJECTIVES = {
    "M-01": "Use the Pythagorean theorem to find a missing side of a right triangle.",
    "S-01": "Describe how adding or removing heat changes the state of a substance.",
    "R-01": "Determine the central idea of a short passage.",
    "R-02":  "Solve a linear equation in one variable."
}

HUMAN_QUEUE = []   # items waiting for a person to review
HELD = []          # items the workflow held back, with the reasons

---
# 4. Build a workflow in three steps

```
 objective
     |
 1. WRITE    the model writes one item
     |
 2. REVIEW   the model scores it against a rubric
     |
 3. CHECK    plain Python rules, no model
     |
 human queue (passed)   or   HELD (failed, with reasons)
```

Our code fixes this order. The model never decides what happens next.

### Step 1: Write

The model fills in a fixed form (`Item`), so every item comes back with the same parts.

Two prompt ideas do most of the work:

- **Wrong answers must be real mistakes.** Each distractor should be something a real student would
  choose, not a random wrong number.
- **Show, don't just tell.** Stating a style rule is often not enough; the model falls back on what it
  saw most in training. Showing a GOOD and a BAD example of each rule is what makes it stick.

In [ ]:
class Item(BaseModel):
    stem: str = Field(description="The question")
    A: str
    B: str
    C: str
    D: str
    key: Literal["A", "B", "C", "D"] = Field(description="The correct option")


WRITER_PROMPT = """You write one grade 8 multiple-choice question for the objective given.
Four options, exactly one correct. Each wrong option must be a mistake a real grade 8 student makes.
For reading, include a short original passage (60-90 words) at the start of the stem.

HOUSE STYLE. Follow the GOOD examples; never produce the BAD ones.
1. The question ends in '?'.
   GOOD: How tall is the pole?
   BAD:  Find the height of the pole.
2. No 'all of the above' or 'none of the above'.
   BAD:  D. All of the above
3. The correct answer is about the same length as the others.
   GOOD: A. 9 cm   B. 12 cm   C. 15 cm   D. 21 cm
   BAD:  C. 15 cm, because 9 squared plus 12 squared is 225
"""

writer = llm.with_structured_output(Item)


def write(objective):
    return writer.invoke([("system", WRITER_PROMPT), ("user", objective)])


def show(item):
    print(item.stem, "\n")
    for letter in "ABCD":
        mark = "*" if letter == item.key else " "
        print(f" {mark} {letter}. {getattr(item, letter)}")


item = safe(write, OBJECTIVES["M-01"])
if item:
    show(item)

Sam is setting up a ladder to reach a high shelf. He places the base of the ladder 5 meters away from the wall, and the ladder itself is 13 meters long. Assuming the ladder, the wall, and the ground form a right triangle, how high up the wall does the ladder reach? 

   A. 8 m
 * B. 12 m
   C. 18 m
   D. 9 m


### Step 2: Review

A second model call scores the item from 1 to 5 on three questions. The item passes only if **every**
score is 3 or more. An average would let one fatal flaw hide behind two good scores.

Treat this reviewer as a **filter, not a judge**. It removes weak items so people spend their time on
good ones. In real use, human experts still reject many items an automated reviewer approves.

In [ ]:
class Review(BaseModel):
    measures_objective: int = Field(ge=1, le=5)
    one_correct_answer: int = Field(ge=1, le=5)
    believable_wrong_answers: int = Field(ge=1, le=5)
    comment: str = Field(description="One sentence on the weakest point")


REVIEWER_PROMPT = ("You review grade 8 test questions. Score each from 1 to 5. Be strict: a "
                   "well-written question with two defensible answers scores 1 on one_correct_answer.")

reviewer = llm.with_structured_output(Review)


def review(item, objective):
    return reviewer.invoke([("system", REVIEWER_PROMPT),
                            ("user", f"Objective: {objective}\n\nItem: {item.model_dump_json()}")])


def low_scores(r):
    """Every score below 3. Empty means the item passes."""
    return {k: v for k, v in r.model_dump().items() if isinstance(v, int) and v < 3}


r = safe(review, item, OBJECTIVES["M-01"]) if item else None
if r:
    print(r.model_dump())
    print("PASSES" if not low_scores(r) else f"FAILS on {low_scores(r)}")

{'measures_objective': 1, 'one_correct_answer': 1, 'believable_wrong_answers': 4, 'comment': "The distractors are not plausibly close to the correct answer, reducing the question's discriminative power."}
FAILS on {'measures_objective': 1, 'one_correct_answer': 1}


### Step 3: Check

Anything a simple rule can decide should be decided by a rule: free, instant, and the same every time.
These three rules match the house style the writer was shown.

In [ ]:
def check_rules(item):
    """House-style rules in plain Python. Returns a list of problems (empty = clean)."""
    problems = []
    options = {letter: getattr(item, letter) for letter in "ABCD"}

    if not item.stem.rstrip().endswith("?"):
        problems.append("the question does not end in '?'")
    for letter, text in options.items():
        if re.search(r"(all|none) of the above", text, re.IGNORECASE):
            problems.append(f"option {letter} says 'of the above'")
    others = [t for l, t in options.items() if l != item.key]
    if len(options[item.key]) > 1.4 * max(len(t) for t in others):
        problems.append("the correct answer is much longer than the others")
    return problems


if item:
    print(check_rules(item) or "clean")

clean


### Put the three steps together

This is the whole workflow: three function calls and one `if`. Passing items go to a person.
Failing items are held **with the reasons**, which is exactly what the agent in Part 5 will use.

In [ ]:
def run_workflow(code):
    objective = OBJECTIVES[code]
    item = write(objective)                       # 1. WRITE
    r = review(item, objective)                   # 2. REVIEW
    problems = check_rules(item)                  # 3. CHECK
    problems += [f"{k} scored {v}" for k, v in low_scores(r).items()]

    if problems:
        HELD.append({"code": code, "item": item, "feedback": "; ".join(problems)})
        print(f"{code}: HELD  ->  {'; '.join(problems)}")
    else:
        HUMAN_QUEUE.append({"code": code, "item": item, "from": "workflow"})
        print(f"{code}: sent to human queue")
    return item


for code in OBJECTIVES:
    safe(run_workflow, code)

M-01: HELD  ->  believable_wrong_answers scored 2
S-01: sent to human queue
R-01: sent to human queue
R-02: sent to human queue


The **human gate** is simply the end of the workflow: nothing goes into the item bank until a person
looks at `HUMAN_QUEUE` and approves it.

---
# 5. Build an agent: the Repair Agent

The workflow holds some items back. Most have **fixable** problems, and fixing is cheaper than
starting over.

Fixing is where an agent earns its place. One item only needs a question mark; another needs a new
wrong answer, then another check, then another fix. **You cannot draw those steps in advance**, so
the model chooses them.

```
held item + feedback
     |
 [agent decides] -> check_rules -> revise_item -> check_rules -> ...
     |
 send_to_human   (its only way out)
```

**Governance in one line:** `ModelCallLimitMiddleware(run_limit=8)` stops the agent after 8 model calls, so a repair that is going nowhere cannot loop forever.


### The tools

A tool is a normal Python function with a docstring. The model reads the docstring to decide when to
use it, so write it like an instruction.

Keep tool arguments **simple and flat** (`stem`, `A`, `B`, ...) rather than one big JSON string;
models fill in flat arguments far more reliably.

In [ ]:
from langchain.tools import tool

WORK = {}   # the item currently being repaired


@tool
def run_checks() -> str:
    """Check the current item against the house-style rules. Returns the problems, or 'clean'."""
    return "; ".join(check_rules(WORK["item"])) or "clean"


@tool
def revise_item(stem: str, A: str, B: str, C: str, D: str) -> str:
    """Replace the current item with your fixed version. Keep the correct answer in the same letter."""
    WORK["item"] = Item(stem=stem, A=A, B=B, C=C, D=D, key=WORK["item"].key)
    return "Item updated. Run the checks again."


@tool
def send_to_human(note: str) -> str:
    """Send the item to a person for review. Use when the checks pass, or when you cannot fix
    something. In the note, say what you changed."""
    HUMAN_QUEUE.append({"code": WORK["code"], "item": WORK["item"], "from": "repair agent",
                        "note": note})
    WORK["sent"] = True
    return "Sent. Stop now."

### The agent

A model, the tools, a short instruction, and a limit. That is all `create_agent` needs.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

repair_agent = create_agent(
    model=llm,
    tools=[run_checks, revise_item, send_to_human],
    system_prompt=("You fix one grade 8 test question so a person can review it. Run the checks, "
                   "fix the problems with revise_item, check again, and when it is clean call "
                   "send_to_human. Keep the objective and the correct answer the same."),
    middleware=[ModelCallLimitMiddleware(run_limit=8)],
)


def repair(code, item, feedback):
    WORK.update(code=code, item=item, sent=False)
    task = (f"Objective: {OBJECTIVES[code]}\nFeedback: {feedback}\n"
            f"Item: {item.model_dump_json()}")
    return repair_agent.invoke({"messages": [("user", task)]})

### A held item to fix

If the workflow held something back, the agent fixes that. If not, it gets this deliberately broken
item: not a question, an "all of the above" option, and a correct answer that gives itself away by
its length.

In [ ]:
BROKEN = Item(
    stem="A ladder is 10 ft long and its base is 6 ft from a wall. Find how high up the wall it reaches",
    A="4 ft",
    B="8 ft, because 10 squared minus 6 squared is 64 and the square root of 64 is 8",
    C="16 ft",
    D="All of the above",
    key="B",
)

if HELD:
    code, target, feedback = HELD[0]["code"], HELD[0]["item"], HELD[0]["feedback"]
else:
    code, target, feedback = "M-01", BROKEN, "; ".join(check_rules(BROKEN))

show(target)
print("\nproblems:", feedback)

A construction crew is setting up a temporary support beam. They place a 13‑foot ladder against a vertical wall so that the base of the ladder is 5 feet away from the wall. The ladder, the wall, and the ground form a right triangle. Using the Pythagorean theorem, how far up the wall does the ladder reach? 

 * A. 12 ft
   B. 8 ft
   C. 18 ft
   D. 10 ft

problems: believable_wrong_answers scored 2


### Run it, and watch what it chooses

Nobody wrote this sequence of steps. The agent picked each one based on the result of the last.

In [ ]:
run = safe(repair, code, target, feedback)

if run:
    print("What the agent chose to do:")
    steps = [c["name"] for m in run["messages"] for c in (getattr(m, "tool_calls", None) or [])]
    for i, name in enumerate(steps, 1):
        print(f"  {i}. {name}")

    print("\nFixed item:\n")
    show(WORK["item"])
    print("\nchecks now:", check_rules(WORK["item"]) or "clean")
    print("sent to a person:", WORK["sent"])

What the agent chose to do:
  1. run_checks
  2. send_to_human

Fixed item:

A construction crew is setting up a temporary support beam. They place a 13‑foot ladder against a vertical wall so that the base of the ladder is 5 feet away from the wall. The ladder, the wall, and the ground form a right triangle. Using the Pythagorean theorem, how far up the wall does the ladder reach? 

 * A. 12 ft
   B. 8 ft
   C. 18 ft
   D. 10 ft

checks now: clean
sent to a person: True


---
## Workflow vs agent: side by side

| | Workflow (Part 4) | Agent (Part 5) |
|---|---|---|
| Who decides the next step | Our code | The model |
| Number of steps | Always three | Unknown until it runs |
| Same result every run? | Close to it | The path can change |
| How we keep it safe | A human queue at the end | A call limit + no bank tool + a human queue |
| Best for | Producing items | Fixing items |

**A sensible progression:** start with good prompts (with examples), add retrieval (RAG) if accuracy
is the problem, and add agents only where a workflow keeps falling short.

## What a real system adds

| This notebook | A production system |
|---|---|
| One writer | A planning step first, then a specialist writer per subject |
| Three rubric scores | Eight or more, including accuracy and bias |
| Three style rules | Many rules, plus shuffling options so the key is not always in one place |
| One human queue | Panel review, then a committee, before pretesting |
| Python lists | An item repository with IDs, timestamps, and a full history |

And the step none of this replaces: **pretesting**. An item is ready when it has field data, not when
a model approves it.

In [ ]:
# YOUR TURN
# 1. Add your own objective to OBJECTIVES and run the workflow on it.
# 2. Add one rule to check_rules (for example: the question must not contain "NOT").
# 3. Send every held item to the Repair Agent and count how many it fixes.

for h in HELD:
    safe(repair, h["code"], h["item"], h["feedback"])
    print(h["code"], "fixed and sent" if WORK.get("sent") else "not fixed")

S-01 fixed and sent
R-01 fixed and sent


In [ ]:
print("=" * 60)
print(f"HUMAN QUEUE: {len(HUMAN_QUEUE)} item(s) waiting for a person")
print("=" * 60)
for i, entry in enumerate(HUMAN_QUEUE, 1):
    print(f"\n[{i}] {entry['code']}  |  from: {entry['from']}")
    if entry.get("note"):
        print(f"    agent's note: {entry['note']}")
    print()
    show(entry["item"])

print("\n" + "=" * 60)
print(f"HELD: {len(HELD)} item(s) the workflow held back (original versions)")
print("=" * 60)
for i, entry in enumerate(HELD, 1):
    print(f"\n[{i}] {entry['code']}  |  why: {entry['feedback']}\n")
    show(entry["item"])

HUMAN QUEUE: 4 item(s) waiting for a person

[1] S-01  |  from: workflow

When ice is left out on a warm day, it begins to melt. The solid ice absorbs heat from the air, causing its particles to move faster and break free from their fixed positions. As the temperature rises further, the liquid water can turn into steam, a gas, if enough heat is added. Conversely, cooling steam can cause it to condense back into liquid water. Which statement best describes what happens to the particles of a substance when heat is added? 

 * A. The particles move faster and spread apart, turning a solid into a liquid or a liquid into a gas.
   B. The particles slow down and come closer together, turning a solid into a liquid.
   C. The particles change shape but keep the same speed, turning a liquid into a solid.
   D. The particles disappear and reappear as a new substance when heated.

[2] R-01  |  from: workflow

Every summer, the town of Willow Creek holds a festival to celebrate the river that runs